In [2]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "sofie"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "sofie_data.db"

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True
)

conn.execute("PRAGMA query_only=ON")

In [3]:
query = """
SELECT
    timestamp,
    p_actual
FROM logging
ORDER BY timestamp
"""

solar = pd.read_sql_query(query, conn)

solar.head()

,timestamp,p_actual
0,2025-12-01 00:00:02.000000,NaN
1,2025-12-01 00:00:07.000000,NaN
2,2025-12-01 00:00:12.000000,NaN
3,2025-12-01 00:00:17.000000,NaN
4,2025-12-01 00:00:22.000000,NaN


In [4]:
solar["timestamp"] = pd.to_datetime(solar["timestamp"])

print("Dataset overview:")
print(solar.info())

print("\nMissing values:")
print(solar["p_actual"].isna().sum())

print("\nValid solar power statistics (W):")
print(solar["p_actual"].describe())

# Missing percentage by hour
solar["hour"] = solar["timestamp"].dt.hour

hourly_missing = solar.groupby("hour")["p_actual"].agg(
    total="count",
    mean="mean"
)

hourly_missing["missing"] = (
    solar.groupby("hour")["p_actual"]
    .apply(lambda x: x.isna().sum())
)

hourly_missing["missing_pct"] = (
    hourly_missing["missing"] /
    (hourly_missing["total"] + hourly_missing["missing"]) * 100
)

hourly_missing

Dataset overview:
<class 'pandas.DataFrame'>
RangeIndex: 3607431 entries, 0 to 3607430
Data columns (total 2 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   p_actual   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 55.0 MB
None

Missing values:
1719844

Valid solar power statistics (W):
count    1.887587e+06
mean     2.438446e+03
std      3.524407e+03
min      0.000000e+00
25%      1.240000e+02
50%      7.670000e+02
75%      3.330000e+03
max      2.020300e+04
Name: p_actual, dtype: float64


,total,mean,missing,missing_pct
hour,,,,
0,1440,74.500000,149050,99.043126
1,1440,74.500000,148872,99.041993
2,2991,31.095286,146776,98.002898
3,32101,78.285162,117666,78.566039
4,54963,408.994305,94801,63.300259
5,77047,975.945475,72716,48.554049
6,100094,2051.004606,49796,33.221696
7,144588,2488.415685,5895,3.917386
8,150259,3196.143559,42,0.027944


In [5]:
# Inspect actual readings during the night
night_data = solar[
    solar["timestamp"].dt.hour.isin([0, 1, 2, 3, 4, 21, 22, 23])
    & solar["p_actual"].notna()
]

print("Nighttime reading statistics:")
print(night_data["p_actual"].describe())

print("\nUnique nighttime values:")
print(sorted(night_data["p_actual"].unique())[:30])

print("\nSample nighttime readings:")
print(night_data[["timestamp", "p_actual"]].head(20))

Nighttime reading statistics:
count    97255.000000
mean       263.451668
std        340.937395
min          0.000000
25%         23.000000
50%        117.000000
75%        394.000000
max       2014.000000
Name: p_actual, dtype: float64

Unique nighttime values:
[np.float64(0.0), np.float64(6.0), np.float64(7.0), np.float64(10.0), np.float64(14.0), np.float64(15.0), np.float64(19.0), np.float64(23.0), np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0), np.float64(31.0), np.float64(32.0), np.float64(33.0), np.float64(34.0), np.float64(35.0), np.float64(36.0), np.float64(37.0), np.float64(38.0), np.float64(39.0), np.float64(40.0), np.float64(41.0), np.float64(42.0), np.float64(43.0), np.float64(44.0), np.float64(45.0), np.float64(46.0), np.float64(47.0), np.float64(48.0)]

Sample nighttime readings:
                 timestamp  p_actual
809886 2026-01-16 21:00:02     117.0
809887 2026-01-16 21:00:07     117.0
809888 2026-01-16 21:00:12     117.0
809889 2026-01-16 21:00

In [6]:
print("Missing values:", solar["p_actual"].isna().sum())

print("Duplicate timestamps:", solar["timestamp"].duplicated().sum())

print("Negative readings:", (solar["p_actual"] < 0).sum())

print("\nSolar power statistics (W):")
print(solar["p_actual"].describe())

Missing values: 1719844
Duplicate timestamps: 3
Negative readings: 0

Solar power statistics (W):
count    1.887587e+06
mean     2.438446e+03
std      3.524407e+03
min      0.000000e+00
25%      1.240000e+02
50%      7.670000e+02
75%      3.330000e+03
max      2.020300e+04
Name: p_actual, dtype: float64


In [7]:
solar["solar_power_kw"] = solar["p_actual"] / 1000

In [8]:
# Average records sharing the same timestamp
solar_by_time = (
    solar.groupby("timestamp")["solar_power_kw"].mean()
)

# Aggregate into 15-minute intervals
solar_15min = solar_by_time.resample("15min").mean()

# Count valid readings in each interval
solar_15min = solar_15min.to_frame(
    name="solar_power_kw"
)

solar_15min["reading_count"] = (
    solar_by_time.resample("15min").count()
)

# Data availability indicator
solar_15min["data_available"] = (
    solar_15min["reading_count"] > 0
)

solar_15min.head()

,solar_power_kw,reading_count,data_available
timestamp,,,
2025-12-01 00:00:00,NaN,0,False
2025-12-01 00:15:00,NaN,0,False
2025-12-01 00:30:00,NaN,0,False
2025-12-01 00:45:00,NaN,0,False
2025-12-01 01:00:00,NaN,0,False


In [9]:
print("Total intervals:", len(solar_15min))

print("\nMissing intervals:")
print(solar_15min["solar_power_kw"].isna().sum())

print("\nReading count statistics:")
print(solar_15min["reading_count"].describe())

print("\nSolar power statistics (kW):")
print(solar_15min["solar_power_kw"].describe())

Total intervals: 20256

Missing intervals:
9519

Reading count statistics:
count    20256.000000
mean        93.186414
std         89.288085
min          0.000000
25%          0.000000
50%        180.000000
75%        180.000000
max        181.000000
Name: reading_count, dtype: float64

Solar power statistics (kW):
count    10737.000000
mean         2.383299
std          3.415970
min          0.000000
25%          0.123044
50%          0.735089
75%          3.320156
max         18.634361
Name: solar_power_kw, dtype: float64


In [10]:
solar_output = solar_15min.reset_index()

output_path = OUTPUT_DIR / "solar_15min.csv"
solar_output.to_csv(output_path, index=False)

print("Exported to:", output_path)
print("Rows:", len(solar_output))
print("Columns:", solar_output.columns.tolist())

Exported to: c:\Users\leena\OneDrive\Desktop\ev-microgrid-twin\data\processed\solar_15min.csv
Rows: 20256
Columns: ['timestamp', 'solar_power_kw', 'reading_count', 'data_available']
